<a href="https://colab.research.google.com/github/kmillaevelyn/data-science-portfolio/blob/main/EMBR3_Previsao_Alta_Baixa_KNN_Streamlit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns
import joblib # Importação para salvar o modelo

from google.colab import files

print("Por favor, faça o upload do arquivo 'EMBR3.SA_historico.csv'")
uploaded = files.upload()

file_name = next(iter(uploaded))

df = pd.read_csv(file_name, skiprows=2)
df.columns = ['Date', 'Close', 'High', 'Low', 'Open', 'Volume']

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

df['Target'] = (df['Close'] > df['Open']).astype(int)

df['High_Low_diff'] = df['High'] - df['Low']
df['Open_Close_diff'] = df['Open'] - df['Close']
df['MA5'] = df['Close'].rolling(window=5).mean()
df['MA20'] = df['Close'].rolling(window=20).mean()
df['Daily_Return'] = df['Close'].pct_change()

df.dropna(inplace=True)
df['Year'] = df['Date'].dt.year

features = ['Open', 'Close', 'High', 'Low', 'High_Low_diff', 'Open_Close_diff', 'MA5', 'MA20', 'Daily_Return']
X = df[features]
y = df['Target']

test_df = df[df['Year'] == 2025].copy()
X_train = X[df['Year'].isin([2023, 2024])]
y_train = y[df['Year'].isin([2023, 2024])]
X_test = test_df[features]
y_test = test_df['Target']


print(f"Tamanho do conjunto de treino (X_train): {X_train.shape}")
print(f"Tamanho do conjunto de teste (X_test): {X_test.shape}")

param_grid = {
    'n_neighbors': range(1, 21),
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

grid_search = GridSearchCV(KNeighborsClassifier(), param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"\n--- Resultados do GridSearchCV com Novas Features ---")
print(f"Melhores parâmetros: {grid_search.best_params_}")
print(f"Melhor acurácia (no conjunto de treino - cross-validation): {grid_search.best_score_:.4f}")

knn_tuned = KNeighborsClassifier(**grid_search.best_params_)
knn_tuned.fit(X_train, y_train)

y_pred_tuned = knn_tuned.predict(X_test)

print("\n--- Avaliação do KNN com parâmetros ajustados e Novas Features ---")
print("Acurácia (ajustada):", accuracy_score(y_test, y_pred_tuned))
print("\nRelatório de Classificação (ajustado):\n", classification_report(y_test, y_pred_tuned))

plt.figure(figsize=(5,4))
sns.heatmap(confusion_matrix(y_test, y_pred_tuned), annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de Confusão - KNN Ajustado (Novas Features)')
plt.xlabel('Previsto')
plt.ylabel('Real')
plt.show()

cm = confusion_matrix(y_test, y_pred_tuned)
tn, fp, fn, tp = cm.ravel()

specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

print(f"\n--- Métricas Adicionais ---")
print(f"Especificidade: {specificity:.4f}")

total_previsoes = len(y_test)
acertos = tp + tn
erros = fp + fn

print(f"\n--- Desempenho de Previsão ---")
print(f"Total de Previsões no Teste: {total_previsoes}")
print(f"Quantidade de Acertos: {acertos}")
print(f"Quantidade de Erros: {erros}")
print(f"Percentual de Acertos: {(acertos / total_previsoes) * 100:.2f}%")
print(f"Percentual de Erros: {(erros / total_previsoes) * 100:.2f}%")

test_df['Previsao'] = y_pred_tuned
test_df['Daily_Trade_Return'] = (test_df['Close'] - test_df['Open']) / test_df['Open']

operacoes_previu_alta = test_df[test_df['Previsao'] == 1]

if not operacoes_previu_alta.empty:
    ganhos = operacoes_previu_alta[operacoes_previu_alta['Target'] == 1]['Daily_Trade_Return'].sum()
    perdas = operacoes_previu_alta[operacoes_previu_alta['Target'] == 0]['Daily_Trade_Return'].sum()
    retorno_geral = ganhos + perdas

    print(f"\n--- Retorno Financeiro Simulado (considerando apenas previsão de 'Alta') ---")
    print(f"Retorno Financeiro dos Ganhos: {ganhos * 100:.2f}%")
    print(f"Retorno Financeiro das Perdas: {perdas * 100:.2f}%")
    print(f"Retorno Financeiro Geral: {retorno_geral * 100:.2f}%")
else:
    print("\n--- Retorno Financeiro Simulado (considerando apenas previsão de 'Alta') ---")
    print("Nenhuma operação de 'Alta' prevista para o período de teste selecionado.")

plt.figure(figsize=(14, 7))
plt.plot(df['Date'], df['Close'], label='Preço de Fechamento')
plt.title('Série Temporal do Preço de Fechamento da EMBR3.SA (2023-2025)')
plt.xlabel('Data')
plt.ylabel('Preço de Fechamento (R$)')
plt.grid(True)
plt.legend()
plt.show()

class_distribution = df['Target'].value_counts(normalize=True) * 100

plt.figure(figsize=(7, 5))
sns.barplot(x=class_distribution.index, y=class_distribution.values, palette='viridis')
plt.title('Distribuição Percentual das Classes (Target)')
plt.xlabel('Classe (0 = Baixa/Igual, 1 = Alta)')
plt.ylabel('Percentual (%)')
plt.xticks(ticks=[0, 1], labels=['0 (Baixa/Igual)', '1 (Alta)'])
plt.show()

print("\n--- Quantitativo em Percentual das Classes Calculadas ---")
print(class_distribution)

# Linha para salvar o modelo treinado (crucial para o dashboard)
joblib.dump(knn_tuned, 'knn_model.pkl')
print("\nModelo KNN treinado salvo como 'knn_model.pkl'")

Por favor, faça o upload do arquivo 'EMBR3.SA_historico.csv'


TypeError: 'NoneType' object is not subscriptable

In [ ]:
%%writefile dashboard_app.py
import streamlit as st
import pandas as pd
import joblib
import plotly.graph_objects as go
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

@st.cache_data
def load_and_process_data(uploaded_file):
    df = pd.read_csv(uploaded_file, skiprows=2)
    df.columns = ['Date', 'Close', 'High', 'Low', 'Open', 'Volume']
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values('Date').reset_index(drop=True)

    df['Target'] = (df['Close'] > df['Open']).astype(int)

    df['High_Low_diff'] = df['High'] - df['Low']
    df['Open_Close_diff'] = df['Open'] - df['Close']
    df['MA5'] = df['Close'].rolling(window=5).mean()
    df['MA20'] = df['Close'].rolling(window=20).mean()
    df['Daily_Return'] = df['Close'].pct_change()

    df.dropna(inplace=True)
    df['Year'] = df['Date'].dt.year
    return df

@st.cache_resource
def load_model():
    model = joblib.load('knn_model.pkl')
    return model

st.set_page_config(layout="wide")
st.title('Dashboard de Previsão de Ações da B3 - EMBR3.SA')

st.sidebar.header('Carregar Dados')
uploaded_file = st.sidebar.file_uploader("Por favor, faça o upload do arquivo 'EMBR3.SA_historico.csv'", type=["csv"])

df_processed = None
if uploaded_file is not None:
    df_processed = load_and_process_data(uploaded_file)
    st.sidebar.success("Dados carregados e processados com sucesso!")
else:
    st.info("Aguardando o upload do arquivo CSV para iniciar a análise.")

if df_processed is not None:
    model = load_model()

    all_years = sorted(df_processed['Year'].unique())
    if len(all_years) >= 2:
        default_train_years = [2023, 2024]
        available_default_train_years = [y for y in default_train_years if y in all_years]
        if not available_default_train_years:
            available_default_train_years = all_years[:2]


        selected_train_years = st.sidebar.multiselect(
            'Selecione os anos para TREINAMENTO:',
            options=all_years,
            default=available_default_train_years
        )
        if len(selected_train_years) == 0:
            st.warning("Por favor, selecione pelo menos um ano para treinamento.")
            st.stop()

        test_year_options = [y for y in all_years if y > max(selected_train_years)]
        if test_year_options:
            selected_test_year = st.sidebar.selectbox(
                'Selecione o ano para TESTE:',
                options=test_year_options,
                index=0
            )
            if selected_test_year in selected_train_years:
                 st.sidebar.warning("O ano de teste não pode ser um ano de treinamento. Por favor, selecione outro.")
                 st.stop()
        else:
            st.warning("Não há anos de teste disponíveis após os anos de treinamento selecionados.")
            st.stop()

        features = ['Open', 'Close', 'High', 'Low', 'High_Low_diff', 'Open_Close_diff', 'MA5', 'MA20', 'Daily_Return']

        X_train_dashboard = df_processed[df_processed['Year'].isin(selected_train_years)][features]
        y_train_dashboard = df_processed[df_processed['Year'].isin(selected_train_years)]['Target']

        test_df_dashboard = df_processed[df_processed['Year'] == selected_test_year].copy()
        X_test_dashboard = test_df_dashboard[features]
        y_test_dashboard = test_df_dashboard['Target']

        if X_test_dashboard.empty:
            st.warning(f"Não há dados para o ano de teste {selected_test_year} após o processamento.")
            st.stop()

        y_pred_dashboard = model.predict(X_test_dashboard)

        st.header(f'Resultados da Previsão para o Ano de Teste: {selected_test_year}')

        accuracy = accuracy_score(y_test_dashboard, y_pred_dashboard)
        st.metric(label="Acurácia do Modelo", value=f"{accuracy*100:.2f}%")

        st.subheader('Relatório de Classificação')
        report_dict = classification_report(y_test_dashboard, y_pred_dashboard, output_dict=True)
        report_df = pd.DataFrame(report_dict).transpose()
        st.dataframe(report_df.style.format("{:.2f}"))

        st.subheader('Matriz de Confusão')
        cm = confusion_matrix(y_test_dashboard, y_pred_dashboard)
        fig_cm = go.Figure(data=go.Heatmap(
                       z=cm,
                       x=['Previsto 0', 'Previsto 1'],
                       y=['Real 0', 'Real 1'],
                       colorscale='Blues',
                       text=cm,
                       texttemplate="%{text}",
                       textfont={"size":20}))
        fig_cm.update_layout(title='Matriz de Confusão')
        st.plotly_chart(fig_cm, use_container_width=True)

        tn, fp, fn, tp = cm.ravel()
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

        st.subheader('Desempenho Detalhado')
        col1, col2, col3 = st.columns(3)
        with col1:
            st.metric(label="Especificidade", value=f"{specificity:.4f}")
        with col2:
            st.metric(label="Quantidade de Acertos", value=f"{tp + tn}")
        with col3:
            st.metric(label="Quantidade de Erros", value=f"{fp + fn}")

        st.write(f"Percentual de Acertos: **{(tp + tn) / len(y_test_dashboard) * 100:.2f}%**")
        st.write(f"Percentual de Erros: **{(fp + fn) / len(y_test_dashboard) * 100:.2f}%**")

        st.subheader('Retorno Financeiro Simulado (Previsão de Alta)')

        test_df_dashboard['Previsao'] = y_pred_dashboard
        test_df_dashboard['Daily_Trade_Return'] = (test_df_dashboard['Close'] - test_df_dashboard['Open']) / test_df_dashboard['Open']

        operacoes_previu_alta = test_df_dashboard[test_df_dashboard['Previsao'] == 1]

        if not operacoes_previu_alta.empty:
            ganhos = operacoes_previu_alta[operacoes_previu_alta['Target'] == 1]['Daily_Trade_Return'].sum()
            perdas = operacoes_previu_alta[operacoes_previu_alta['Target'] == 0]['Daily_Trade_Return'].sum()
            retorno_geral = ganhos + perdas

            col_ret1, col_ret2, col_ret3 = st.columns(3)
            with col_ret1:
                st.metric(label="Retorno Financeiro Ganhos", value=f"{ganhos * 100:.2f}%")
            with col_ret2:
                st.metric(label="Retorno Financeiro Perdas", value=f"{perdas * 100:.2f}%")
            with col_ret3:
                st.metric(label="Retorno Financeiro Geral", value=f"{retorno_geral * 100:.2f}%")
        else:
            st.info("Nenhuma operação de 'Alta' prevista para o período de teste selecionado.")

        st.header('Análise da Série Temporal e Classes')
        st.subheader('Preço de Fechamento da EMBR3.SA ao longo do Tempo')

        fig_ts = go.Figure(data=[go.Scatter(x=df_processed['Date'], y=df_processed['Close'], mode='lines', name='Preço de Fechamento')])
        fig_ts.update_layout(
            title='Série Temporal do Preço de Fechamento da EMBR3.SA (2023-2025)',
            xaxis_title='Data',
            yaxis_title='Preço de Fechamento (R$)',
            hovermode="x unified"
        )
        st.plotly_chart(fig_ts, use_container_width=True)

        st.subheader('Distribuição Percentual das Classes (Target)')
        class_distribution = df_processed['Target'].value_counts(normalize=True) * 100

        fig_bar = go.Figure(data=[go.Bar(x=['0 (Baixa/Igual)', '1 (Alta)'], y=class_distribution.values, marker_color=['lightcoral', 'lightgreen'])])
        fig_bar.update_layout(
            title='Distribuição Percentual das Classes (Target)',
            xaxis_title='Classe',
            yaxis_title='Percentual (%)'
        )
        st.plotly_chart(fig_bar, use_container_width=True)

        st.subheader('Quantitativo em Percentual das Classes Calculadas')
        st.write(class_distribution)

    else:
        st.warning("Dados insuficientes para seleção de anos de treino/teste. Certifique-se de ter dados de pelo menos 2 anos.")

Writing dashboard_app.py


In [ ]:
!pip install pyngrok streamlit

import os
from pyngrok import ngrok
from google.colab import userdata

ngrok.kill()

# Seu token do ngrok está inserido aqui:
ngrok.set_auth_token(userdata.get('NGROK_AUTH_TOKEN'))

!nohup streamlit run dashboard_app.py &

url_publica = ngrok.connect(addr=8501, proto="http")
print(f"Seu dashboard Streamlit está disponível em: {url_publica}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 66.2 MB/s eta 0:00:00


SecretNotFoundError: Secret NGROK_AUTH_TOKEN does not exist.